# 05.03 — Profile vs. Profile

Orthograph's `compare_profiles(left, right)` runs a **symmetric diff** between
two `GraphProfile` objects. It answers the question:

> *What structural differences exist between these two observed graph snapshots?*

All emitted issues have `Severity.INFO`. The comparison is direction-neutral:
something present only in `left` gets `*_ONLY_IN_LEFT`; something only in
`right` gets `*_ONLY_IN_RIGHT`.

This notebook:
1. Builds two `GraphProfile` objects manually (no database required).
2. Calls `compare_profiles(a, b)` and inspects the `ValidationResult`.
3. Shows how to filter by diff code and display the results.

In [ ]:
from orthograph.comparison.engine import compare_profiles

## 1. Build two profiles

Imagine the same Neo4j database inspected at two points in time (or two
different environments). We construct the profiles manually here to keep the
notebook self-contained.

**Profile A** — baseline snapshot with Person, Movie, City and ACTED_IN, LIVES_IN.

**Profile B** — later snapshot: City is gone, a new Genre label appeared, and
the `name` property type on Person changed from `String` to `Integer` in
one environment.

In [ ]:
from shared.profiles import FILMOGRAPHY_PROFILE, FILMOGRAPHY_PROFILE_STAGING


# profile_a — production baseline (Person, Movie, City + ACTED_IN, DIRECTED, LIVES_IN)
profile_a = FILMOGRAPHY_PROFILE

# profile_b — staging snapshot: City gone, Genre added, Person.name type changed,
#              LIVES_IN removed, IN_GENRE added, ACTED_IN cardinality min raised.
profile_b = FILMOGRAPHY_PROFILE_STAGING

print("Profile A nodes:", sorted(profile_a.node_labels))
print("Profile B nodes:", sorted(profile_b.node_labels))

## 2. Run the comparison

`compare_profiles` returns a `ValidationResult`. Since both sides are profiles,
it uses `diff_rules()` by default — all issues are `INFO` and the result is
always `is_valid = True`.

In [ ]:
result = compare_profiles(profile_a, profile_b)

print(f"Total diff issues : {len(result.issues)}")
print(f"is_valid          : {result.is_valid}  (always True for diff comparisons)")
print(f"Errors            : {len(result.errors)}")

## 3. Inspect all diff issues

In [ ]:
for issue in result.issues:
    print(f"[{issue.severity.value.upper():4}] {issue.code:<30} {issue.entity_id}")
    print(f"       {issue.message}")
    if issue.context:
        print(f"       context: {issue.context}")
    print()

## 4. Filter by diff code

Use `.issues` and filter on `.code` to focus on a specific diff category.

In [ ]:
# Labels that exist only in profile_a (left)
left_only_nodes = [i for i in result.issues if i.code == "NODE_LABEL_ONLY_IN_LEFT"]
right_only_nodes = [i for i in result.issues if i.code == "NODE_LABEL_ONLY_IN_RIGHT"]

print("Node labels only in A:", [i.entity_id for i in left_only_nodes])
print("Node labels only in B:", [i.entity_id for i in right_only_nodes])

In [ ]:
# Property type changes
type_changes = [i for i in result.issues if i.code == "PROPERTY_TYPE_CHANGED"]
for issue in type_changes:
    print(
        f"{issue.entity_id}: {issue.context.get('left')} → {issue.context.get('right')}"
    )

In [ ]:
# Cardinality changes
card_changes = [i for i in result.issues if i.code == "CARDINALITY_CHANGED"]
for issue in card_changes:
    ctx = issue.context
    print(
        f"{issue.entity_id}: left min={ctx.get('left_min')} max={ctx.get('left_max')} | right min={ctx.get('right_min')} max={ctx.get('right_max')}"
    )

## 5. Identical profiles → zero issues

When comparing a profile against itself, no differences exist.

In [ ]:
import copy

In [ ]:
profile_a_copy = copy.deepcopy(profile_a)
assert profile_a_copy is not profile_a

In [ ]:
self_result = compare_profiles(profile_a, profile_a)
print(f"Issues when comparing profile_a to itself: {len(self_result.issues)}")
assert self_result.issues == [], "Expected zero issues"